In [13]:
%pip install -q psycopg2-binary boto3

import io
import json
import psycopg2
import boto3

config_db = {"host": "postgres", "database": "postgres", "user": "postgres", "password": "senha123", "port": 5432}
config_s3 = {"endpoint_url": "http://minio:9000", "aws_access_key_id": "root-minio", "aws_secret_access_key": "root12345678", "region_name": "local"}

bucket_name = "raw"
destino_s3 = "python_native/telemetria_jsonl/telemetria_clickstream.jsonl"

Note: you may need to restart the kernel to use updated packages.


In [14]:
#Extração (Extract) do PostgreSQL com Tipo Complexo (JSONB)
print("Extraindo eventos de telemetria com cargas JSONB...")

with psycopg2.connect(**config_db) as conn:
    with conn.cursor() as cursor:
        cursor.execute("SELECT id_evento, timestamp_evento, tipo_evento, payload FROM eventos_web;")
        registros_banco = cursor.fetchall()

print(f"Quantidade de eventos coletados: {len(registros_banco)}")

Extraindo eventos de telemetria com cargas JSONB...
Quantidade de eventos coletados: 50000


In [23]:
#Serialização e Carga (Load) para o MinIO via JSON Lines
print("Serializando registros para o formato JSON Lines em memoria...")
buffer_memoria = io.StringIO()

for linha in registros_banco:
    registro = {
        "id_evento": str(linha[0]),
        "timestamp_evento": str(linha[1]),
        "tipo_evento": linha[2],
        "payload_original": linha[3]
    }
    buffer_memoria.write(json.dumps(registro, ensure_ascii=False) + "\n")

print(f"Realizando upload para s3://{bucket_name}/{destino_s3}...")
s3_client = boto3.client('s3', **config_s3)
s3_client.put_object(
    Bucket=bucket_name,
    Key=destino_s3,
    Body=buffer_memoria.getvalue(),
    ContentType='application/x-ndjson'
)

print("Ingestao JSONL concluida com sucesso!")

Serializando registros para o formato JSON Lines em memoria...
Realizando upload para s3://raw/python_native/telemetria_jsonl/telemetria_clickstream.jsonl...
Ingestao JSONL concluida com sucesso!


In [24]:
# Encerramento das Conexoes
print("Fechando a conexao com o banco de dados PostgreSQL...")
conn.close()
print("Conexao encerrada com seguranca. Os recursos do banco foram liberados.")

Fechando a conexao com o banco de dados PostgreSQL...
Conexao encerrada com seguranca. Os recursos do banco foram liberados.


### Analise de Arquitetura: O Formato JSON Lines (JSONL) no Python Nativo

**Caracteristicas Principais:** Formato baseado em texto plano, semi-estruturado, focado em registros por linhas.

#### Vantagens
* **Esquema Flexivel:** Se a estrutura interna do payload mudar dinamicamente na origem, o JSONL absorve os novos campos sem quebrar o historico.
* **Suporte a Aninhamento:** Armazena sub-objetos, listas e arrays complexos perfeitamente.

#### Desvantagens
* **Redundancia Massiva:** O nome das colunas e chaves se repete textualmente em todas as linhas, gerando desperdicio de bytes armazenados.

#### Caso de Uso no Mercado
* Camada Landing ou Raw (Bronze) para recepcao de dados brutos provenientes de Webhooks, APIs REST externas e logs de auditoria de servidores.